# 01. Data Understanding — SECOM Semiconductor Manufacturing Data

## 목적
이 노트북의 목표는 **모델을 만드는 것**이 아니라, 분석 전에 데이터의 구조와 품질을 파악하는 것입니다.

확인할 항목:
1. 데이터 크기와 변수 수
2. Pass / Fail 비율
3. 결측치
4. 중복 샘플
5. 상수 변수 / 거의 변하지 않는 변수
6. 이후 분석에서 우선 정리할 변수 후보

> **중요:** SECOM의 공정 변수는 익명화되어 있으므로, 각 Parameter의 실제 물리적 의미를 임의로 추정하지 않습니다.


## 0. 데이터 파일 준비

프로젝트 폴더를 아래처럼 구성합니다.

```text
semiconductor-yield-troubleshooting/
├─ data/
│  └─ raw/
│     ├─ secom.data
│     └─ secom_labels.data
├─ notebooks/
│  └─ 01_data_understanding.ipynb
└─ README.md
```

SECOM 원본 데이터는 UCI Machine Learning Repository에서 받을 수 있습니다.

- `secom.data`: 공정 변수
- `secom_labels.data`: Pass/Fail label + timestamp


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

print("Libraries loaded.")


## 1. 파일 경로 설정

In [ ]:
# 노트북을 notebooks/ 폴더에서 실행하는 경우와
# 프로젝트 루트에서 실행하는 경우를 모두 처리합니다.

cwd = Path.cwd()
PROJECT_DIR = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR = PROJECT_DIR / "data" / "raw"
FEATURE_PATH = DATA_DIR / "secom.data"
LABEL_PATH = DATA_DIR / "secom_labels.data"

print("PROJECT_DIR:", PROJECT_DIR)
print("FEATURE_PATH exists:", FEATURE_PATH.exists())
print("LABEL_PATH exists:", LABEL_PATH.exists())

if not FEATURE_PATH.exists() or not LABEL_PATH.exists():
    raise FileNotFoundError(
        "data/raw 폴더에 secom.data와 secom_labels.data를 넣어주세요."
    )


## 2. 데이터 불러오기

SECOM 공정 변수에는 원래 변수명이 없으므로 `Parameter_000`, `Parameter_001`처럼 이름을 붙입니다.


In [ ]:
# 공정 변수
X = pd.read_csv(
    FEATURE_PATH,
    sep=r"\s+",
    header=None
)

X.columns = [f"Parameter_{i:03d}" for i in range(X.shape[1])]

# Label + timestamp
labels = pd.read_csv(
    LABEL_PATH,
    sep=r"\s+",
    header=None,
    names=["label", "date", "time"]
)

labels["timestamp"] = pd.to_datetime(
    labels["date"] + " " + labels["time"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

labels = labels[["label", "timestamp"]]

print("X shape:", X.shape)
print("labels shape:", labels.shape)

display(X.head())
display(labels.head())


## 3. 기본 구조 확인

여기서는 다음을 확인합니다.

- 샘플 수
- 공정 변수 수
- Label 값
- Timestamp 파싱 여부


In [ ]:
print(f"샘플 수: {X.shape[0]:,}")
print(f"공정 변수 수: {X.shape[1]:,}")
print("\nLabel unique values:", sorted(labels["label"].dropna().unique()))
print("Timestamp missing:", labels["timestamp"].isna().sum())

assert len(X) == len(labels), "공정 데이터와 Label 행 수가 다릅니다."


## 4. Pass / Fail 비율 확인

SECOM 데이터에서:

- `-1` = Pass
- `1` = Fail

제조 데이터에서는 Fail이 상대적으로 적을 수 있으므로 **단순 Accuracy만으로 모델 성능을 판단하면 안 됩니다.**
이 불균형 정도를 먼저 확인합니다.


In [ ]:
label_map = {-1: "Pass", 1: "Fail"}

label_counts = (
    labels["label"]
    .map(label_map)
    .value_counts()
    .rename_axis("result")
    .reset_index(name="count")
)

label_counts["ratio_pct"] = (
    label_counts["count"] / label_counts["count"].sum() * 100
).round(2)

display(label_counts)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(label_counts["result"], label_counts["count"])
ax.set_title("Pass / Fail Distribution")
ax.set_xlabel("Result")
ax.set_ylabel("Count")
plt.show()


### 해석할 것
실행 후 아래 질문에 답합니다.

- Fail 비율은 몇 %인가?
- 클래스 불균형이 심한가?
- 이후 모델링에서 Accuracy보다 Recall, F1, Balanced Accuracy 등을 같이 봐야 하는가?

이 단계에서는 **결론을 미리 쓰지 말고 실제 출력값을 보고 기록**합니다.


## 5. 결측치 분석

In [ ]:
missing_count = X.isna().sum()
missing_ratio = X.isna().mean() * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_ratio_pct": missing_ratio
}).sort_values("missing_ratio_pct", ascending=False)

print("결측치가 하나라도 있는 변수 수:",
      (missing_summary["missing_count"] > 0).sum())

print("결측률 50% 이상 변수 수:",
      (missing_summary["missing_ratio_pct"] >= 50).sum())

display(missing_summary.head(20))


In [ ]:
top_missing = missing_summary.head(20).sort_values("missing_ratio_pct")

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top_missing.index, top_missing["missing_ratio_pct"])
ax.set_title("Top 20 Parameters by Missing Ratio")
ax.set_xlabel("Missing ratio (%)")
ax.set_ylabel("Parameter")
plt.show()


### 결측치에서 바로 삭제하지 않는 이유

결측치가 많다고 해서 반드시 쓸모없는 변수는 아닙니다.

예를 들어 실제 제조현장에서는:
- 특정 Tool에서만 측정되는 변수
- 특정 Recipe에서만 생성되는 변수
- 특정 시점 이후 추가된 Sensor

일 수도 있습니다.

SECOM은 익명 데이터이므로 실제 이유를 확인할 수 없습니다.  
따라서 여기서는 **결측률이 높은 변수를 후보로 표시만 하고**, 최종 제거 기준은 다음 분석 단계에서 결정합니다.


## 6. 상수 변수 확인

In [ ]:
nunique = X.nunique(dropna=True)

constant_cols = nunique[nunique <= 1].index.tolist()

print("상수 변수 수:", len(constant_cols))
print("예시:", constant_cols[:20])


상수 변수는 모든 샘플에서 값이 동일하므로 Pass/Fail 차이를 설명할 수 없습니다.  
따라서 이후 분석에서 제거할 수 있는 가장 명확한 후보입니다.


## 7. 거의 변하지 않는 변수 확인

In [ ]:
def dominant_value_ratio(series):
    non_null = series.dropna()
    if len(non_null) == 0:
        return np.nan
    return non_null.value_counts(normalize=True, dropna=False).iloc[0]

dominant_ratio = X.apply(dominant_value_ratio)

near_constant_summary = pd.DataFrame({
    "n_unique": nunique,
    "dominant_value_ratio": dominant_ratio
}).sort_values("dominant_value_ratio", ascending=False)

# 99% 이상 같은 값을 가지는 변수를 'near-constant 후보'로 표시
near_constant_cols = near_constant_summary[
    (near_constant_summary["dominant_value_ratio"] >= 0.99)
    & (near_constant_summary["n_unique"] > 1)
].index.tolist()

print("Near-constant 후보 수:", len(near_constant_cols))
display(near_constant_summary.head(20))


`99%`는 절대적인 정답이 아니라 **탐색용 기준**입니다.

이 프로젝트에서는 이 변수를 자동 삭제하기보다,
> "정보량이 매우 낮아 추가 검토가 필요한 변수"

로 기록하는 편이 좋습니다.


## 8. 중복 샘플 확인

In [ ]:
duplicate_count = X.duplicated().sum()

print("완전히 동일한 공정 데이터 행 수:", duplicate_count)


중복이 있다고 해서 바로 삭제하지 않습니다.

실제 제조 데이터에서는 서로 다른 생산 entity가 같은 측정값을 가질 가능성도 있기 때문입니다.  
SECOM에는 Lot ID / Tool ID 같은 식별 정보가 없으므로 이 단계에서는 **중복 여부만 기록**합니다.


## 9. 기본 통계량 확인

In [ ]:
basic_stats = X.describe().T

display(
    basic_stats[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
    .head(20)
)


## 10. 데이터 품질 요약

지금까지 확인한 내용을 한 번에 정리합니다.


In [ ]:
fail_count = int((labels["label"] == 1).sum())
pass_count = int((labels["label"] == -1).sum())

summary = pd.DataFrame({
    "metric": [
        "samples",
        "features",
        "pass_count",
        "fail_count",
        "fail_ratio_pct",
        "features_with_missing",
        "features_missing_ge_50pct",
        "constant_features",
        "near_constant_features",
        "duplicate_rows"
    ],
    "value": [
        len(X),
        X.shape[1],
        pass_count,
        fail_count,
        round(fail_count / len(labels) * 100, 2),
        int((missing_count > 0).sum()),
        int((missing_ratio >= 50).sum()),
        len(constant_cols),
        len(near_constant_cols),
        int(duplicate_count)
    ]
})

display(summary)


## 11. 이후 제거 검토 대상 정리

이 Notebook에서는 실제 삭제를 많이 하지 않습니다.  
우선 **명백한 상수 변수**와 **결측률이 매우 높은 변수**를 검토 대상으로 정리합니다.


In [ ]:
high_missing_cols = missing_summary[
    missing_summary["missing_ratio_pct"] >= 50
].index.tolist()

review_cols = sorted(set(constant_cols + high_missing_cols))

print("우선 검토 대상 변수 수:", len(review_cols))
print(review_cols[:30])


# 12. Engineering Takeaway

노트북 실행 후 아래 내용을 **직접 출력값을 보고** 작성합니다.

예시 형식:

> 본 데이터는 총 ___개의 생산 샘플과 ___개의 익명화된 공정 Parameter로 구성되어 있다.  
> Fail은 ___건으로 전체의 ___%를 차지해 클래스 불균형이 확인되었다.  
> 또한 ___개의 변수에서 결측치가 존재하고, ___개의 상수 변수가 확인되었다.  
> 따라서 다음 단계에서는 데이터 품질이 낮은 변수를 정리한 뒤 Pass/Fail 간 Parameter 분포 차이를 분석한다.

## 다음 Notebook
`02_eda.ipynb`

핵심 질문:

> **"Pass 샘플과 Fail 샘플은 어떤 Parameter에서 가장 큰 차이를 보이는가?"**
